# پیشرفت پایان‌نامه — پیاده‌سازی کامل روش مقاله (برای نمایش به استاد)

**وضعیت:** این نوت‌بوک هر ۴ جزء زیر را به‌صورت واقعی (نه شبیه‌سازی) اجرا می‌کند:

1. **BERT خالص** (baseline) — ستون "BERT" جدول ۹ مقاله
2. **BERT + BiLSTM** — مدل پیشنهادی مقاله
3. **جدول مقایسه** — معادل جدول ۱۰ مقاله (آیا BiLSTM واقعاً کمک می‌کند؟)
4. **Apriori** — بخش ۳.۲/۳.۳ مقاله (استخراج و همبستگی عوامل علّی)

**نکته‌ی مهم:** پیش‌فرض پایین `FAST_DEMO = False` است — یعنی مستقیم با تنظیمات دقیق جدول ۹
مقاله (۲۰ epoch، effective batch size ۳۲ با gradient accumulation، مناسب یک GPU لپ‌تاپی
۴ گیگ‌بایتی مثل GTX 1650Ti) اجرا می‌شود. این همان عددی است که باید در فصل نتایج پایان‌نامه
بیاید، نه یک پیش‌نمایش. با توجه به یک روز وقتی که دارید، همین الان Run All بزنید و بگذارید
در پس‌زمینه اجرا شود (تخمین: حدود ۱-۲ ساعت به‌ازای هر مدل، جمعاً چند ساعت).

Run this notebook from inside the `bert-bilstm-marine-accidents/` folder of the repo so the
relative imports below (`dataset.py`, `model.py`, `baseline_model.py`, `causal_factors.py`,
`training_utils.py`) resolve correctly.

---

Reference paper: Zhao, Z.; Liu, X.; Feng, L.; Grifoll, M.; Feng, H. (2025). *Causation Analysis
of Marine Traffic Accidents Using Deep Learning Approaches: A Case Study from China's Coasts.*
Systems, 13(4), 284. https://doi.org/10.3390/systems13040284
(full text in the repo root: `Causation_Analysis_of_Marine_Traffic_Accidents_Usi.pdf`)

Dataset: [`baker-street/maib-incident-reports-5K`](https://huggingface.co/datasets/baker-street/maib-incident-reports-5K)
(UK MAIB marine incident narratives, labeled by incident type).


## 0. Setup

Install once (skip if already installed in your environment):


In [ ]:
# !pip install -q torch transformers datasets scikit-learn mlxtend pandas matplotlib seaborn tabulate

import os
import glob

# On Colab, this notebook is often opened as its own kernel *after* cloning/unzipping
# the repo in a different notebook/cell -- that clone lives on disk, but a fresh kernel
# still starts in /content, so the plain imports below would fail with
# "ModuleNotFoundError: No module named 'dataset'". This finds the project folder
# (wherever it was cloned/unzipped under /content) and cds into it. If you're already
# running from inside bert-bilstm-marine-accidents/ (e.g. locally), this is a no-op.
if not os.path.exists("dataset.py"):
    candidates = [
        c for c in (
            sorted(glob.glob("/content/**/bert-bilstm-marine-accidents", recursive=True))
            + sorted(glob.glob("**/bert-bilstm-marine-accidents", recursive=True))
        )
        if os.path.exists(os.path.join(c, "dataset.py"))
    ]
    if candidates:
        os.chdir(candidates[0])
    else:
        raise FileNotFoundError(
            "Can't find bert-bilstm-marine-accidents/ (with dataset.py) anywhere under "
            f"the current directory ({os.getcwd()}) or /content. Clone/unzip the repo in "
            "THIS SAME notebook/kernel first (a separate notebook tab starts a fresh "
            "kernel at /content and won't see files from another notebook's clone/unzip) "
            "-- see the project README for the exact commands."
        )
print("Working directory:", os.getcwd())

import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import BertTokenizerFast

sys.path.insert(0, os.getcwd())  # make sure the project's own .py modules are importable

from dataset import MAIBTextDataset, load_maib_splits, clean_text, DATASET_NAME
from model import BertBiLSTMClassifier
from baseline_model import BertClassifier
from training_utils import run_epoch, set_seed
from causal_factors import extract_causal_factors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 1. Config

With a full day available, run the **paper-faithful settings directly** (`FAST_DEMO = False`,
the default below): 20 epochs, effective batch size 32 via gradient accumulation (Table 9 /
Section 4.1). These are the numbers that belong in the thesis's results chapter, not a
rough preview. `FAST_DEMO = True` is kept only as a quick sanity check (5 epochs) if you
want to confirm the pipeline runs end-to-end before committing to the long run.

| | FAST_DEMO=True (sanity check only) | FAST_DEMO=False (default — real thesis numbers) |
|---|---|---|
| Epochs | 5 | 20 (Figure 7: converges by ~20) |
| Batch size (per step) | 8 | 8 (same — GPU memory limited) |
| Grad accumulation steps | 4 (effective batch 32, matches Table 9) | 4 (effective batch 32, matches Table 9) |
| Expected wall time (GTX 1650Ti, 4GB) | ~15-20 min per model | Roughly 1-2 hours per model (estimate — start it and let it run) |

Start both trainings as early in the day as you can (each model is a separate cell below) —
they run sequentially, so budget a few hours total, and there's no harm leaving the notebook
running in the background while you work on the thesis text.


In [ ]:
FAST_DEMO = False  # sanity-check only; keep this False for the real thesis run

class Config:
    dataset_name = DATASET_NAME
    bert_name = "bert-base-uncased"
    max_length = 128

    # BiLSTM branch (Table 9, "BERT + BiLSTM" column)
    lstm_hidden = 128
    lstm_layers = 1

    # Baseline BERT branch (Table 9, "BERT" column)
    baseline_hidden_dim = 512

    dropout = 0.3
    batch_size = 8 if FAST_DEMO else 8          # kept small for 4GB VRAM either way
    grad_accum_steps = 4                        # 8 x 4 = effective batch 32, matches Table 9
    epochs = 5 if FAST_DEMO else 20              # Figure 7: paper converges by ~20 epochs

    lr = 1e-6                # Table 9 (both columns use the same rate)
    l2_weight_decay = 0.05   # Table 9, both columns
    l1_lambda_baseline = 1e-8    # Table 9, "BERT" column
    l1_lambda_bilstm = 5e-10     # Table 9, "BERT + BiLSTM" column
    max_grad_norm_baseline = 2.35  # Table 9, "BERT" column
    max_grad_norm_bilstm = 2.75    # Table 9, "BERT + BiLSTM" column

    val_size = 0.15          # Section 4.1: 70/15/15 split
    test_size = 0.15
    seed = 42


cfg = Config()
set_seed(cfg.seed)
use_amp = device.type == "cuda"
print(f"FAST_DEMO={FAST_DEMO} | epochs={cfg.epochs} | batch_size={cfg.batch_size} "
      f"| grad_accum_steps={cfg.grad_accum_steps} (effective batch {cfg.batch_size * cfg.grad_accum_steps}) "
      f"| mixed precision: {use_amp}")


## 2. Load and split the dataset (Section 4.1: 70/15/15, stratified)

In [ ]:
splits, label_encoder = load_maib_splits(seed=cfg.seed, val_size=cfg.val_size, test_size=cfg.test_size)
classes = list(label_encoder.classes_)
num_classes = len(classes)

print(f"{num_classes} classes: {classes}")
for name, (texts, labels) in splits.items():
    print(f"  {name}: {len(texts)} examples")

tokenizer = BertTokenizerFast.from_pretrained(cfg.bert_name)

train_ds = MAIBTextDataset(*splits["train"], tokenizer, cfg.max_length)
val_ds = MAIBTextDataset(*splits["val"], tokenizer, cfg.max_length)
test_ds = MAIBTextDataset(*splits["test"], tokenizer, cfg.max_length)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)


## 3. Part 1 — Plain BERT baseline (Table 9, "BERT" column)

The paper trains this alongside BERT + BiLSTM specifically to measure what the BiLSTM stage
adds (its own Table 10: 88.7% vs. 89.8% on the paper's data). Architecture: BERT's pooled
`[CLS]` output &rarr; 2-layer, 512-unit GELU MLP head, 3 dropout layers (see
`baseline_model.py` for exactly why this layout, since Table 9 names the width and dropout
count but not the arrangement).


In [ ]:
def train_model(model, name, l1_lambda, max_grad_norm):
    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.l2_weight_decay)
    scaler = torch.amp.GradScaler(device=device.type, enabled=use_amp)

    history = {"train_loss": [], "train_acc": [], "train_f1": [],
               "val_loss": [], "val_acc": [], "val_f1": []}
    best_val_f1, best_state = -1.0, None
    t0 = time.time()
    for epoch in range(1, cfg.epochs + 1):
        train_loss, train_acc, train_f1, _, _ = run_epoch(
            model, train_loader, device, optimizer, max_grad_norm, l1_lambda,
            cfg.grad_accum_steps, scaler,
        )
        val_loss, val_acc, val_f1, _, _ = run_epoch(model, val_loader, device)
        print(f"[{name}] epoch {epoch}/{cfg.epochs} | train_loss={train_loss:.4f} "
              f"train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f} "
              f"val_f1={val_f1:.4f}")

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["train_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"[{name}] training done in {(time.time() - t0) / 60:.1f} min, best val_f1={best_val_f1:.4f}")
    return model, history


def evaluate_model(model, name):
    _, acc, macro_f1, preds, labels = run_epoch(model, test_loader, device)
    report_text = classification_report(
        labels, preds, labels=list(range(num_classes)), target_names=classes,
        digits=4, zero_division=0,
    )
    report_dict = classification_report(
        labels, preds, labels=list(range(num_classes)), target_names=classes,
        digits=4, zero_division=0, output_dict=True,
    )
    print(f"\n=== {name}: test accuracy={acc:.4f}  macro-F1={macro_f1:.4f} ===")
    print(report_text)
    return report_dict, preds, labels


In [ ]:
baseline_model = BertClassifier(
    num_classes=num_classes,
    bert_name=cfg.bert_name,
    hidden_dim=cfg.baseline_hidden_dim,
    dropout=cfg.dropout,
).to(device)

baseline_model, baseline_history = train_model(
    baseline_model, "BERT baseline",
    l1_lambda=cfg.l1_lambda_baseline, max_grad_norm=cfg.max_grad_norm_baseline,
)
baseline_report, baseline_preds, baseline_labels = evaluate_model(baseline_model, "BERT baseline")

os.makedirs("checkpoints_bert_baseline", exist_ok=True)
torch.save(baseline_model.state_dict(), "checkpoints_bert_baseline/best_model.pt")
with open("checkpoints_bert_baseline/label_classes.json", "w") as f:
    json.dump(classes, f)


## 4. Part 2 — BERT + BiLSTM (the paper's proposed model, Table 9 second column)

BERT encoder &rarr; dropout &rarr; 128-unit Bidirectional LSTM &rarr; concat(final forward,
final backward hidden states) &rarr; Mish &rarr; dropout &rarr; linear &rarr; softmax
(Figure 1 / Section 3.1 of the paper). See `model.py` (`BertBiLSTMClassifier`).


In [ ]:
bilstm_model = BertBiLSTMClassifier(
    num_classes=num_classes,
    bert_name=cfg.bert_name,
    lstm_hidden=cfg.lstm_hidden,
    lstm_layers=cfg.lstm_layers,
    dropout=cfg.dropout,
).to(device)

bilstm_model, bilstm_history = train_model(
    bilstm_model, "BERT + BiLSTM",
    l1_lambda=cfg.l1_lambda_bilstm, max_grad_norm=cfg.max_grad_norm_bilstm,
)
bilstm_report, bilstm_preds, bilstm_labels = evaluate_model(bilstm_model, "BERT + BiLSTM")

os.makedirs("checkpoints", exist_ok=True)
torch.save(bilstm_model.state_dict(), "checkpoints/best_model.pt")
with open("checkpoints/label_classes.json", "w") as f:
    json.dump(classes, f)


## 4b. Training curves (paper's Figure 7 style)

Loss and accuracy per epoch, train vs. validation, for both models. Useful evidence in the
thesis that training actually converged rather than just quoting a final number.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, metric, ylabel in [(axes[0], "loss", "Loss"), (axes[1], "acc", "Accuracy")]:
    epochs_range = range(1, cfg.epochs + 1)
    ax.plot(epochs_range, baseline_history[f"train_{metric}"], "--", color="#4C72B0", label="BERT (train)")
    ax.plot(epochs_range, baseline_history[f"val_{metric}"], "-", color="#4C72B0", label="BERT (val)")
    ax.plot(epochs_range, bilstm_history[f"train_{metric}"], "--", color="#DD8452", label="BERT+BiLSTM (train)")
    ax.plot(epochs_range, bilstm_history[f"val_{metric}"], "-", color="#DD8452", label="BERT+BiLSTM (val)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} per epoch")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()


## 4c. Confusion matrices (test set)

Per-class error pattern for both models on the same held-out test split.


In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, preds, labels, title in [
    (axes[0], baseline_preds, baseline_labels, "BERT (baseline)"),
    (axes[1], bilstm_preds, bilstm_labels, "BERT + BiLSTM"),
]:
    cm = confusion_matrix(labels, preds, labels=list(range(num_classes)))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(num_classes))
    ax.set_yticks(range(num_classes))
    ax.set_xticklabels(classes, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(classes, fontsize=7)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=7)

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()


## 5. Part 3 — Comparison table (Table 10 style ablation): does BiLSTM help?

Both models were just evaluated on the **same** held-out test split. This reproduces, on
this project's dataset, the ablation the paper runs on its own dataset in Table 10.


In [ ]:
summary_rows = [
    ("Accuracy", baseline_report["accuracy"], bilstm_report["accuracy"]),
    ("Macro avg F1", baseline_report["macro avg"]["f1-score"], bilstm_report["macro avg"]["f1-score"]),
    ("Weighted avg F1", baseline_report["weighted avg"]["f1-score"], bilstm_report["weighted avg"]["f1-score"]),
]

comparison_df = pd.DataFrame(
    [(m, b, s, s - b) for m, b, s in summary_rows],
    columns=["Metric", "BERT", "BERT + BiLSTM", "Delta"],
)
comparison_df


In [ ]:
per_class_rows = [
    (cls, baseline_report[cls]["f1-score"], bilstm_report[cls]["f1-score"],
     bilstm_report[cls]["f1-score"] - baseline_report[cls]["f1-score"])
    for cls in classes
]
per_class_df = pd.DataFrame(per_class_rows, columns=["Class", "BERT", "BERT + BiLSTM", "Delta"])
per_class_df


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
metrics = comparison_df["Metric"]
x = np.arange(len(metrics))
width = 0.35
ax.bar(x - width / 2, comparison_df["BERT"], width, label="BERT")
ax.bar(x + width / 2, comparison_df["BERT + BiLSTM"], width, label="BERT + BiLSTM")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("Score")
ax.set_title("BERT vs. BERT + BiLSTM (Table 10 style ablation)")
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("comparison_table10_style.png", dpi=150)
plt.show()


## 6. Part 4 — Apriori: mining causal-factor associations (Section 3.2/3.3, Tables 5-8)

**Important caveat, be upfront about this with your advisor:** the paper mines its own
hand-labeled, multi-label causal-factor tags (a human expert tagged each report with one or
more of 32 categories). The MAIB dataset used here only has a single *incident-type* label
per report — there is nothing to mine co-occurrence between directly. `causal_factors.py`
works around this by heuristically re-deriving multi-label causal-factor tags per report via
keyword/phrase matching against the paper's own 32 category names (its Table 10 lists them
verbatim). This demonstrates the Apriori pipeline end-to-end and produces plausible-looking
associations, but it is **not a reproduction of the paper's manual annotation** — treat it as
a stand-in, not as validated re-labeling of this dataset.


In [ ]:
from datasets import load_dataset
from mlxtend.frequent_patterns import apriori, association_rules

raw = load_dataset(DATASET_NAME, split="train")
all_texts = [clean_text(t) for t in raw["text"]]

factor_lists = [extract_causal_factors(t) for t in all_texts]
n_tagged = sum(1 for f in factor_lists if f)
avg_factors = sum(len(f) for f in factor_lists) / len(factor_lists)
print(f"Keyword-tagged {n_tagged}/{len(all_texts)} reports with >=1 causal factor "
      f"(avg {avg_factors:.2f} factors/report)")

all_categories = sorted({c for factors in factor_lists for c in factors})
print(f"{len(all_categories)} distinct causal-factor categories matched at least once")

onehot = pd.DataFrame(
    [{c: (c in factors) for c in all_categories} for factors in factor_lists]
)

MIN_SUPPORT = 0.008     # matches the paper's own general-rule threshold (Section 4.4)
MIN_CONFIDENCE = 0.15

frequent_itemsets = apriori(onehot, min_support=MIN_SUPPORT, use_colnames=True)
print(f"{len(frequent_itemsets)} frequent itemsets at min_support={MIN_SUPPORT}")

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules["rule"] = rules.apply(
    lambda r: f"{', '.join(sorted(r['antecedents']))} -> {', '.join(sorted(r['consequents']))}",
    axis=1,
)
print(f"{len(rules)} association rules at min_confidence={MIN_CONFIDENCE}")


In [ ]:
top_by_confidence = rules.sort_values("confidence", ascending=False).head(10)[
    ["rule", "support", "confidence", "lift"]
].reset_index(drop=True)
print("Top rules by confidence (paper's Tables 5/6 style):")
top_by_confidence


In [ ]:
top_by_lift = rules.sort_values("lift", ascending=False).head(10)[
    ["rule", "support", "confidence", "lift"]
].reset_index(drop=True)
print("Top rules by lift (paper's Tables 7/8 style):")
top_by_lift


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_data = top_by_lift.iloc[::-1]  # largest lift at top of the horizontal bar chart
ax.barh(plot_data["rule"], plot_data["lift"], color="#4C72B0")
ax.set_xlabel("Lift")
ax.set_title("Top causal-factor association rules by lift (Apriori)")
plt.tight_layout()
plt.savefig("apriori_top_rules_by_lift.png", dpi=150)
plt.show()


## 7. Final summary: all four components in one table

This is the single table to put in front of your advisor — the four components of the
paper's method, each with its headline result on this project's dataset.


In [ ]:
summary = pd.DataFrame([
    {
        "Component": "1. BERT (baseline)",
        "Paper reference": "Table 9 'BERT' column",
        "Headline result": f"Accuracy={baseline_report['accuracy']:.4f}, "
                            f"Macro-F1={baseline_report['macro avg']['f1-score']:.4f}",
    },
    {
        "Component": "2. BERT + BiLSTM",
        "Paper reference": "Table 9 'BERT + BiLSTM' column / Figure 1",
        "Headline result": f"Accuracy={bilstm_report['accuracy']:.4f}, "
                            f"Macro-F1={bilstm_report['macro avg']['f1-score']:.4f}",
    },
    {
        "Component": "3. Comparison (ablation)",
        "Paper reference": "Table 10",
        "Headline result": f"BiLSTM changes accuracy by "
                            f"{bilstm_report['accuracy'] - baseline_report['accuracy']:+.4f} "
                            f"and macro-F1 by "
                            f"{bilstm_report['macro avg']['f1-score'] - baseline_report['macro avg']['f1-score']:+.4f}",
    },
    {
        "Component": "4. Apriori (causal factors)",
        "Paper reference": "Section 3.2/3.3, Tables 5-8",
        "Headline result": f"{len(all_categories)} categories matched, "
                            f"{len(frequent_itemsets)} frequent itemsets, "
                            f"{len(rules)} rules "
                            f"(top rule: {top_by_lift.iloc[0]['rule']}, lift={top_by_lift.iloc[0]['lift']:.2f})",
    },
])
summary


In [ ]:
with open("thesis_progress_summary.md", "w") as f:
    f.write(f"# Thesis progress summary ({'FAST_DEMO' if FAST_DEMO else 'FULL PAPER-FAITHFUL'} run)\n\n")
    f.write(summary.to_markdown(index=False))
    f.write("\n\n## BERT vs. BERT+BiLSTM (Table 10 style)\n\n")
    f.write(comparison_df.to_markdown(index=False))
    f.write("\n\n## Per-class F1\n\n")
    f.write(per_class_df.to_markdown(index=False))
    f.write("\n\n## Top Apriori rules by confidence\n\n")
    f.write(top_by_confidence.to_markdown(index=False))
    f.write("\n\n## Top Apriori rules by lift\n\n")
    f.write(top_by_lift.to_markdown(index=False))
print("Saved thesis_progress_summary.md -- everything above in one Markdown file, ready to paste into your thesis draft or an email to your advisor.")


## نکته‌ی پایانی

- با `FAST_DEMO = False` (پیش‌فرض)، اعداد بالا، نمودارهای همگرایی (بخش ۴b) و ماتریس درهم‌ریختگی
  (بخش ۴c) همان چیزی هستند که باید در فصل نتایج پایان‌نامه بیایند — نه یک پیش‌نمایش موقت.
- محدودیت شناخته‌شده که باید به استاد گفت: دیتاست این پروژه (`incident-type`, ۵ کلاس عمومی)
  با دیتاست خود مقاله (۳۲ کلاس عامل علّی، augment‌شده) فرق دارد، پس درصد دقت قابل مقایسه‌ی
  مستقیم با عدد ۸۹.۸٪ مقاله نیست — معماری و هایپرپارامترها دقیقاً از مقاله گرفته شده‌اند،
  اما داده متفاوت است. بخش Apriori هم روی برچسب‌های علّی *بازسازی‌شده با کلیدواژه* اجرا شده،
  نه برچسب دستی مقاله — این را هم شفاف بگویید.
- خروجی `thesis_progress_summary.md` و چهار تصویر (`training_curves.png`,
  `confusion_matrices.png`, `comparison_table10_style.png`, `apriori_top_rules_by_lift.png`)
  همه بعد از اجرای کامل نوت‌بوک در همین پوشه ذخیره می‌شوند — مستقیم قابل استفاده در متن
  پایان‌نامه یا اسلاید.
